In [5]:
"""
02 - Filter speeches that mention climate, and explore Arctic mentions.

Goal: produce a smaller filtered CSV with only climate-related speeches,
plus understand whether Arctic / climate-security mentions are frequent enough
to warrant a focused angle.
"""

from pathlib import Path
import sys
import pandas as pd

# Resolve project root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Force reimport in case modules are edited
for mod in ["src.text_processing", "src.data_loader"]:
    if mod in sys.modules:
        del sys.modules[mod]

from src.text_processing import (
    CLIMATE_KEYWORDS,
    ALL_CLIMATE_KEYWORDS,
    add_keyword_flags,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load the full corpus
print("Loading full corpus...")
df = pd.read_csv(PROCESSED_DIR / "all_speeches.csv")
print(f"  {len(df):,} speeches loaded")

# Add boolean flags for each climate theme
print("\nAnnotating with theme flags (this takes ~30 seconds for 609k rows)...")
df = add_keyword_flags(df, themes=CLIMATE_KEYWORDS, text_col="text")

# Summary: how many speeches mention any climate theme?
print(f"\nSpeeches mentioning ANY climate theme: {df['is_climate_any'].sum():,} ({df['is_climate_any'].mean()*100:.1f}%)")

print("\nBy theme:")
for theme in CLIMATE_KEYWORDS:
    col = f"is_climate_{theme}"
    n = df[col].sum()
    print(f"  {theme:<20} {n:>10,} ({n/len(df)*100:.2f}%)")

Loading full corpus...
  609,209 speeches loaded

Annotating with theme flags (this takes ~30 seconds for 609k rows)...


c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df[text_col].fillna("").str.contains(pattern, regex=True)
c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df[text_col].fillna("").str.contains(pattern, regex=True)
c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df[text_col].fillna("").str.contains(pattern, regex=True)
c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: Th


Speeches mentioning ANY climate theme: 22,088 (3.6%)

By theme:
  core                     13,488 (2.21%)
  energy_transition        11,518 (1.89%)
  policy                    3,179 (0.52%)
  security_nexus              718 (0.12%)
  arctic                       72 (0.01%)


In [6]:
"""
Save the climate-related subset and inspect a few examples.
This is the dataset I'll use for all downstream analysis.
"""

# Filter to only climate-related speeches
df_climate = df[df["is_climate_any"]].copy()

# Sort by date for readability
df_climate = df_climate.sort_values("date").reset_index(drop=True)

print(f"Climate speeches: {len(df_climate):,}")
print(f"Period: {df_climate['date'].min()} to {df_climate['date'].max()}")

# Save to CSV
CLIMATE_CSV = PROCESSED_DIR / "climate_speeches.csv"
df_climate.to_csv(CLIMATE_CSV, index=False, encoding="utf-8")
print(f"\nSaved to: {CLIMATE_CSV}")
print(f"File size: {CLIMATE_CSV.stat().st_size / (1024*1024):.1f} MB")

# Quick distributions on the climate subset
print(f"\nClimate speeches per year:")
print(df_climate["year"].value_counts().sort_index())

print(f"\nTop 10 parties on climate:")
print(df_climate["party_abbr"].value_counts().head(10))

Climate speeches: 22,088
Period: 2014-04-16 to 2022-07-12

Saved to: c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\data\processed\climate_speeches.csv
File size: 56.3 MB

Climate speeches per year:
year
2014     156
2015    2223
2016    2042
2017    2548
2018    3616
2019    4207
2020    2344
2021    2960
2022    1992
Name: count, dtype: int64

Top 10 parties on climate:
party_abbr
VVD     4524
D66     3539
PvdA    2313
GL      2198
CDA     2017
PvdD    1782
PVV     1369
CU      1296
SP      1290
FvD      465
Name: count, dtype: int64


In [7]:
"""
Validate keyword matches: look at examples of speeches matching
'security_nexus' and 'arctic' to make sure the keywords work as intended.
"""

print("=" * 70)
print("EXAMPLES: Climate-security nexus speeches")
print("=" * 70)
security_speeches = df_climate[df_climate["is_climate_security_nexus"]]
print(f"Total: {len(security_speeches):,}")
print()

# Show 3 examples (truncated)
for i, row in security_speeches.sample(3, random_state=42).iterrows():
    print(f"[{row['date']}] {row['speaker_name']} ({row['party_abbr']}):")
    print(f"  {row['text'][:400]}...")
    print()

print("=" * 70)
print("EXAMPLES: Arctic-related speeches")
print("=" * 70)
arctic_speeches = df_climate[df_climate["is_climate_arctic"]]
print(f"Total: {len(arctic_speeches):,}")
print()

# Show 3 examples
for i, row in arctic_speeches.sample(3, random_state=42).iterrows():
    print(f"[{row['date']}] {row['speaker_name']} ({row['party_abbr']}):")
    print(f"  {row['text'][:400]}...")
    print()

# Quick: top years for Arctic mentions
print("Arctic mentions per year:")
print(arctic_speeches["year"].value_counts().sort_index())

EXAMPLES: Climate-security nexus speeches
Total: 718

[2016-11-22] Jaco Geurts (CDA):
  Mr. Chairman . The CDA is proud of Dutch farmers , gardeners and fishermen , in the hei- and the side - form . The CDA wants agriculture to have a future in the Netherlands . This is important for our food security and because agriculture makes an important contribution to our economy . After four years of policy of this VVD - PvdA cabinet , the CDA sees the following . There is no additional effo...

[2019-02-13] Jaco Geurts (CDA):
  Mr. Chairman . On 20 December last , the government wrote to the Chamber that the drought in the Netherlands was officially over . There was no longer any threat of water shortage . There was a lot of rain in the weeks and months before both the Netherlands and Germany and France . As a result , the water levels in the Rhine and the Meuse are back on track . So this debate also has characteristics...

[2015-06-10] Jaco Geurts (CDA):
  Mr. Chairman . In our society , th

In [8]:
"""
Diagnostic: which Arctic keywords are matching, and on what context?
make sure not to get false positives.
"""

import re
from src.text_processing import _build_keyword_pattern, CLIMATE_KEYWORDS

# Build the Arctic pattern
arctic_pattern = _build_keyword_pattern(CLIMATE_KEYWORDS["arctic"])

# For each Arctic-tagged speech, find which keywords actually matched
arctic_matches = []
for _, row in arctic_speeches.iterrows():
    matches = arctic_pattern.findall(row["text"])
    arctic_matches.append([m.lower() for m in matches])

# Flatten and count
from collections import Counter
flat_matches = [m for sublist in arctic_matches for m in sublist]
print("Distribution of which Arctic keywords actually matched:")
for kw, count in Counter(flat_matches).most_common():
    print(f"  {kw:<25} {count}")

# Look at the suspicious ones — let's see context
print("\n" + "=" * 70)
print("Sample contexts for each Arctic keyword (to spot false positives):")
print("=" * 70)
for kw in CLIMATE_KEYWORDS["arctic"]:
    pat = re.compile(r"\b" + re.escape(kw) + r"\b", re.IGNORECASE)
    matching = arctic_speeches[arctic_speeches["text"].str.contains(pat, regex=True, na=False)]
    if len(matching) == 0:
        continue
    print(f"\n--- Keyword: '{kw}' ({len(matching)} speeches) ---")
    sample = matching.iloc[0]
    text = sample["text"]
    # Find the keyword and show 80 chars before/after
    m = pat.search(text)
    if m:
        start = max(0, m.start() - 80)
        end = min(len(text), m.end() + 80)
        snippet = text[start:end]
        print(f"  ...{snippet}...")

Distribution of which Arctic keywords actually matched:
  arctic                    40
  greenland                 13
  permafrost                13
  spitsbergen               9
  ice sheet                 5
  arctic council            3
  melting ice               3
  sea ice                   2
  antarctic                 2
  ice cap                   2
  northern sea route        1

Sample contexts for each Arctic keyword (to spot false positives):

--- Keyword: 'arctic' (32 speeches) ---
  ...No , I did n't make that cover . One time I 'm in the Chamber and I hear that Arctic oil is n't allowed . The other time I hear that we should not depend on gas fro...

--- Keyword: 'antarctic' (2 speeches) ---
  ...I propose to add to the Chamber 's agenda : the Bill Amendment of the Antarctic Protection Act to extend access to specially protected areas , extend licensing...

--- Keyword: 'ice cap' (3 speeches) ---
  ... practical complications come right around the corner . The same applies

In [9]:
# Reload module after edit
for mod in ["src.text_processing"]:
    if mod in sys.modules:
        del sys.modules[mod]
from src.text_processing import CLIMATE_KEYWORDS, add_keyword_flags

# Re-annotate
df = add_keyword_flags(df.drop(columns=[c for c in df.columns if c.startswith("is_climate")]), 
                       themes=CLIMATE_KEYWORDS, text_col="text")

# Re-summary
print(f"Climate ANY: {df['is_climate_any'].sum():,}")
for theme in CLIMATE_KEYWORDS:
    n = df[f'is_climate_{theme}'].sum()
    print(f"  {theme:<20} {n:>10,}")

# Save updated climate subset
df_climate = df[df["is_climate_any"]].copy().sort_values("date").reset_index(drop=True)
df_climate.to_csv(PROCESSED_DIR / "climate_speeches.csv", index=False, encoding="utf-8")
print(f"\nSaved {len(df_climate):,} climate speeches.")

c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df[text_col].fillna("").str.contains(pattern, regex=True)
c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df[text_col].fillna("").str.contains(pattern, regex=True)
c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = df[text_col].fillna("").str.contains(pattern, regex=True)
c:\Users\becla\Documents\projects\hcss-parlamint-climate-security\src\text_processing.py:148: UserWarning: Th

Climate ANY: 22,088
  core                     13,488
  energy_transition        11,518
  policy                    3,179
  security_nexus              718
  arctic                       72

Saved 22,088 climate speeches.
